> **Open in Google Colab.** The next cell installs `coptpy`. The bundled free license is size-limited but covers this case. Then run *Runtime > Run all*.


In [ ]:
# COPT Python API and this case's dependencies
!pip install -q coptpy


# Expansion 2: Changes in optional course time periods

Since the morning time period is selectable, the range of optional courses is expanded, the test data will change, and an entry will be added to the dictionary object accordingly:

``('CIS 102', 1): [4.8, [('Tues', 5)]],``

In [1]:
from coptpy import *

# Create environment
env = Envr()

# Create model 
model = env.createModel(name="course_scheduling")

# Data
instructor, instructorRating, instructorTime = multidict({
    ('MGT 490', 1): [4.3, [('Mon', 4)]],
    ('MGT 490', 2): [3.8, [('Tues', 4)]],
    ('MGT 490', 3): [3.5, [('Wed', 4)]],
    ('MGT 490', 4): [3.5, [('Fri', 4)]],
    ('MGT 490', 5): [4.6, [('Mon', 1), ('Wed', 2)]],
    ('MGT 490', 6): [2.7, [('Tues', 2), ('Thur', 1)]],
    ('FIN 358', 1): [3.5, [('Wed', 4)]],
    ('FIN 358', 2): [3.3, [('Tues', 2), ('Thur', 1)]],
    ('CIS 102T', 1): [4.4, [('Wed', 3)]],
    ('CIS 102T', 2): [3.1, [('Thur', 3)]],
    ('CIS 102W', 1): [3.7, [('Tues', 4)]],
    ('CIS 102W', 2): [3.5, [('Wed', 3)]],
    ('CIS 102', 1): [4.8, [('Tues', 5)]],
    ('FIN 325', 1): [3.0, [('Thur', 4)]],
    ('FIN 325', 2): [3.7, [('Mon', 1), ('Wed', 2)]],
    ('FIN 352', 1): [3.6, [('Mon', 4)]],
    ('FIN 352', 2): [3.9, [('Mon', 2), ('Wed', 1)]],
    ('FIN 356', 1): [3.2, [('Tues', 4)]],
    ('FIN 356', 2): [3.4, [('Tues', 2), ('Thur', 1)]],
    ('FIN 359', 1): [3.0, [('Mon', 4)]],
    ('FIN 359', 2): [3.5, [('Wed', 4)]],
}) 


Cardinal Optimizer v8.0.5. Build date May 30 2026
Copyright Cardinal Operations 2026. All Rights Reserved



# Code

## Modeling and solving

If you choose the course ``('CIS 102', 1)`` (taking up its morning time slot), a new requirement will be added: ``('Mon', 4)`` and ``('Tues ', 4)`` Courses cannot be selected during these two time periods, which will cause some new constraints to be added to the model.

The code implementation of other modeling parts is consistent with the original model and will not be described again.

In [3]:
weekdays = ['Mon', 'Tues', 'Wed', 'Thur', 'Fri']
timeInstructor = dict()
for w in weekdays:
    for j in [1,2,3,4,5]:
        timeInstructor[(w, j)] = []
for i in instructor:
    for t in instructorTime[i]:
        timeInstructor[t].append(i)
        
compulsoryCourse = ['MGT 490', 'FIN 358']
cisOptionalCourse = ['CIS 102T', 'CIS 102W', 'CIS 102']
finOptionalCourse = ['FIN 325', 'FIN 352', 'FIN 356', 'FIN 359']
optionalCourse = cisOptionalCourse + finOptionalCourse

# Create environment
env = Envr()

# Create model 
model = env.createModel(name="course_scheduling")

x = model.addVars(instructor, vtype=COPT.BINARY)

model.addConstrs(quicksum(x[i] for i in instructor if i[0] == c) == 1 for c in compulsoryCourse)
model.addConstrs(quicksum(x[i] for i in instructor if i[0] == c) <= 1 for c in optionalCourse) 
model.addConstr(quicksum(x[i] for i in instructor for c in cisOptionalCourse if i[0] == c) == 1)
model.addConstr(quicksum(x[i] for i in instructor for c in finOptionalCourse if i[0] == c) == 2)

model.addConstrs(quicksum(x[i] for i in timeInstructor[t]) <= 1 for t in timeInstructor.keys())
model.addConstrs(quicksum(x[i] for i in timeInstructor[(k,1)] + timeInstructor[(k,2)]) <= 1 for k in weekdays)
model.addConstrs(quicksum(x[i] for i in timeInstructor[(k,2)] + timeInstructor[(k,3)]) <= 1 for k in weekdays)

model.setObjective(quicksum(instructorRating[i] * x[i] for i in instructor), sense=COPT.MAXIMIZE)

Cardinal Optimizer v8.0.5. Build date May 30 2026
Copyright Cardinal Operations 2026. All Rights Reserved



The new constraints are of **if-then** type, namely:

$$
\text{if }x=1, \text{then } y=0
$$

Since all variables in this problem are 0-1, they can be expressed as follows:

$$
y <= 1-x
$$

In [5]:
model.addConstrs(x[i] <= 1 - x[('CIS 102', 1)] for k in ['Mon', "Tues"] for i in timeInstructor[(k,4)])
model.solve()

Model fingerprint: 63860483

Using Cardinal Optimizer v8.0.5 on Windows (22H2 Build 19045 - x86_64)
The CPU model is Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz
Hardware has 4 physical cores and 8 logical cores. Using instruction set X86_AVX2 (10)
Maximizing a MIP problem

The original problem has:
    52 rows, 21 columns and 94 non-zero elements
    21 binaries

Starting the MIP solver with 8 threads and 32 tasks

Presolving the problem

The presolved problem has:
    12 rows, 14 columns and 38 non-zero elements
    14 binaries

Problem info:
    Range of matrix coefficients:    [1e+00,1e+00]
    Range of rhs coefficients:       [1e+00,2e+00]
    Range of bound coefficients:     [1e+00,1e+00]
    Range of cost coefficients:      [3e-01,4e+00]
    Density of cost:                     100.0%

     Nodes    Active  LPit/n  IntInf     BestBound  BestSolution     Gap   Time
         0         1      --       0  4.300000e+01            --     Inf  0.05s
H        0         1      --       0  4

## Get and print results

After printing the solution results, we found that the original optimal solution has not changed, and 'CIS 102' does not appear in the course selection list.

In [7]:
timeCode = {
    1: "1:25-2:20 p.m.",
    2: "1:25-3:15 p.m.",
    3: "2:30-5:15 p.m.",
    4: "6:00-8:45 p.m.",
    5: "9:05-11:50 a.m."
}

if model.status == COPT.OPTIMAL:
    print("-"*58)
    print("|     course\t |     rating\t|         time\t\t |")
    for i in instructor:
        if x[i].x >= 0.9:
            timeList = instructorTime[i]
            print("-"*58)
            print("|    {0}\t |      {1}\t| {2}  {3}\t |".format(i[0], instructorRating[i], timeList[0][0], timeCode[timeList[0][1]]))
            if len(timeList) > 1:
                print("|\t\t |\t\t| {2}  {3}\t |".format(i[0], instructorRating[i], timeList[1][0], timeCode[timeList[1][1]]))
    print("-"*58)

----------------------------------------------------------
|     course	 |     rating	|         time		 |
----------------------------------------------------------
|    MGT 490	 |      4.3	| Mon  6:00-8:45 p.m.	 |
----------------------------------------------------------
|    FIN 358	 |      3.5	| Wed  6:00-8:45 p.m.	 |
----------------------------------------------------------
|    CIS 102T	 |      4.4	| Wed  2:30-5:15 p.m.	 |
----------------------------------------------------------
|    FIN 352	 |      3.9	| Mon  1:25-3:15 p.m.	 |
|		 |		| Wed  1:25-2:20 p.m.	 |
----------------------------------------------------------
|    FIN 356	 |      3.4	| Tues  1:25-3:15 p.m.	 |
|		 |		| Thur  1:25-2:20 p.m.	 |
----------------------------------------------------------
